# 14 · Evaluating existing models

A model does not have to be trained in the run that evaluates it. This guide
saves two WarpRec models, evaluates them again from their files, and puts a
recommendation list computed outside WarpRec next to them. It covers:

- what a saved model holds, and how the train pipeline saves one
  (`meta.save_model`, `writer.save_split`);
- the evaluation pipeline with `meta.load_from`: new metrics and cut-offs
  without training, with the same numbers as the model in memory, and what
  loading requires;
- `ProxyRecommender`: evaluating a file another framework wrote, its format,
  and what happens to users and items the file leaves out or WarpRec does not
  know;
- `meta.save_recs`: the file WarpRec writes, how `mask_seen` shapes it, and
  reading it back;
- significance tests between saved models and an external one.

Training is guide 9's subject and the `Evaluator` guide 12's; this guide reuses
both.

**Requirements:** `pip install warprec jupyter`. Runs in about two minutes on
MovieLens-100K (`guide_data.movielens_100k()` downloads and converts it exactly
as guide 1 shows).

In [1]:
import logging
import shutil
import sys
import warnings
from pathlib import Path

import lightning as L
import numpy as np
import pandas as pd
import torch
import yaml
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

folder = movielens_100k()
RUNS = Path("../runs/14-evaluating-existing-models")
shutil.rmtree(RUNS, ignore_errors=True)
RUNS.mkdir(parents=True)

logger.disable("warprec")
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")
warnings.filterwarnings("ignore", module="torch")

## 1. Models to evaluate

The data and the two models come from one configuration: BPR, trained for a
few epochs, and EASE, which is fitted in closed form. The test set is each
user's latest 10% of interactions.

In [2]:
print(Path("configs/design.yml").read_text())

# The data and the two models to train. Every later configuration reads the
# split this run writes, so every evaluation sees exactly the same test set.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  BPR:
    embedding_size: 32
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 10
    learning_rate: 0.005
  EASE:
    l2: 200.0
evaluation:
  top_k: [10]
  metrics: [nDCG, Recall]



In [3]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration

config = load_design_configuration("configs/design.yml")
dataset, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
print(dataset.get_dims(), "users x items")

(943, 1648) users x items


### Writing the split

Every evaluation below has to see the same test set, so the split is written
to disk first and every later configuration reads it back. The writer's
`write_tabular_split` is what `writer.save_split: true` calls in the train
pipeline; it writes `train.tsv` and `test.tsv` under
`<local_experiment_path>/<dataset_name>/split/`.

In [4]:
from warprec.data.writer import LocalWriter

writer = LocalWriter(dataset_name="ml-100k", local_path=str(RUNS / "trained"))
writer.write_tabular_split(dataset, None, None)

split_dir = RUNS / "trained/ml-100k/split"
{p.name: len(pd.read_csv(p, sep="\t")) for p in sorted(split_dir.iterdir())}

{'test.tsv': 10401, 'train.tsv': 89561}

### Training and scoring in memory

BPR is trained the way guide 9 does it, through a Lightning `Trainer`; EASE is
fitted by its constructor. The `Evaluator` scores both in memory. These are
the numbers every reloaded copy below has to reproduce.

In [5]:
from warprec.evaluation import Evaluator
from warprec.recommenders.trainer.runtime import lightning_runtime
from warprec.utils.helpers import model_param_from_dict
from warprec.utils.registry import model_registry


def build(name, ds=dataset, **overrides):
    """Build a model of config.models with its hyperparameters, on ds."""
    params = {**config.models[name], **overrides}
    seed = model_param_from_dict(name, params).optimization.properties.seed
    return model_registry.get(
        name, params=params, info=ds.info(), interactions=ds.train_set, seed=seed
    )


def evaluate(model, ds=dataset, cutoffs=(10,), metrics=("nDCG", "Recall")):
    """Evaluate a model on the test set of ds; one number per metric@k."""
    evaluator = Evaluator(
        list(metrics), list(cutoffs), train_set=ds.train_set.get_sparse()
    )
    evaluator.evaluate(model, ds.get_evaluation_dataloader(), "full", ds)
    return {
        f"{metric}@{k}": value.nanmean().item()
        for k, values in evaluator.compute_results().items()
        for metric, value in values.items()
    }


bpr = build("BPR")
optimization = model_param_from_dict("BPR", config.models["BPR"]).optimization
bpr.set_optimization_parameters(optimization.optimizer, optimization.lr_scheduler)
L.Trainer(
    max_epochs=bpr.epochs,
    accelerator="cpu",
    **lightning_runtime(optimization, "cpu"),
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
).fit(
    bpr,
    train_dataloaders=bpr.get_dataloader(
        dataset.train_set, dataset.train_session, num_workers=0
    ),
)
ease = build("EASE")

in_memory = pd.DataFrame({"BPR": evaluate(bpr), "EASE": evaluate(ease)}).T
in_memory

,nDCG@10,Recall@10
BPR,0.122330,0.106430
EASE,0.154209,0.139149


### Saving the models

`write_model` is what `meta.save_model: true` calls. It writes
`serialized/<name_param>.pth`, where `name_param` is the model's name followed
by a readable name derived from a hash of its hyperparameters: the same
hyperparameters always give the same file name.

In [6]:
for model in (bpr, ease):
    writer.write_model(model, dataset)

serialized = RUNS / "trained/ml-100k/serialized"
print(bpr.name_param, ease.name_param)
{p.name: f"{p.stat().st_size / 1e6:.1f} MB" for p in sorted(serialized.iterdir())}

BPR_NippyValiantNewt EASE_WiseFoxOfForce


{'BPR_NippyValiantNewt.pth': '0.8 MB', 'EASE_WiseFoxOfForce.pth': '32.9 MB'}

The file is a dictionary saved with `torch.save`, built by the model's
`get_state()`. Next to the model's name, hyperparameters and `state_dict`, it
keeps `info` (the dataset's dimensions and **id mappings**), `artifacts` (the
model's plain attributes) and, because the dataset was passed to
`write_model`, a `serving` payload with each user's seen items, which is what
guide 17 serves from.

In [7]:
bpr_file = torch.load(serialized / f"{bpr.name_param}.pth", weights_only=False)
ease_file = torch.load(serialized / f"{ease.name_param}.pth", weights_only=False)
pd.DataFrame(
    {
        name: {
            "keys": ", ".join(state),
            "state_dict": ", ".join(state["state_dict"]) or "(empty)",
            "info": f"{state['info']['n_users']} users, {state['info']['n_items']} items",
            "serving": ", ".join(state["serving"]),
        }
        for name, state in [("BPR", bpr_file), ("EASE", ease_file)]
    }
)

,BPR,EASE
keys,"name, params, info, state_dict, artifacts, for...","name, params, info, state_dict, artifacts, for..."
state_dict,"user_embedding.weight, item_embedding.weight",(empty)
info,"943 users, 1648 items","943 users, 1648 items"
serving,"seen, histories, context_maps, context_stats, ...","seen, histories, context_maps, context_stats, ..."


BPR's result lives in its embedding parameters, so `state_dict` holds it. EASE
keeps its item-item matrix in a plain attribute, so its `state_dict` is empty
and the dense 1,648 × 1,648 matrix travels in `artifacts`, which makes its
file the larger of the two.

In a configuration, the split and the models are written by the train
pipeline (guide 10) when these keys are set; `meta.save_recs` is section 5's
subject. This guide does not run it: the API calls above are what it does.

In [8]:
print(Path("configs/train.yml").read_text())

# The same step as a train-pipeline run (guide 10): save_split writes the split,
# meta.save_model writes serialized/<name>.pth, meta.save_recs writes recs/.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
writer:
  dataset_name: ml-100k
  writing_method: local
  local_experiment_path: ../runs/14-evaluating-existing-models/trained
  save_split: true
models:
  BPR:
    embedding_size: 32
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 10
    learning_rate: 0.005
    meta: {save_model: true, save_recs: true}
  EASE:
    l2: 200.0
    meta: {save_model: true, save_recs: true}
evaluation:
  top_k: [10]
  metrics: [nDCG, Recall]
general:
  ray_address: local
  ray_verbose: 0



## 2. Loading a saved model in Python

`from_checkpoint` is the class method that rebuilds a model from that
dictionary. BPR is rebuilt from its hyperparameters and `info`, then its
`state_dict` is loaded. EASE needs the training interactions to be built, so
when none are passed it is restored from its `artifacts`, without fitting.

In [9]:
bpr_loaded = model_registry.get_class("BPR").from_checkpoint(bpr_file)
ease_loaded = model_registry.get_class("EASE").from_checkpoint(ease_file)

users = torch.arange(dataset.get_dims()[0])
with torch.no_grad():
    print(
        "same scores:",
        torch.equal(bpr.predict(users), bpr_loaded.predict(users)),
        torch.equal(ease.predict(users), ease_loaded.predict(users)),
    )
pd.DataFrame({"BPR": evaluate(bpr_loaded), "EASE": evaluate(ease_loaded)}).T.equals(
    in_memory
)

same scores: True True


True

## 3. The evaluation pipeline with `meta.load_from`

The evaluation pipeline (`-p eval` from the command line, `eval_pipeline` in
Python) reads the data, builds every model in `models` and evaluates it; it
never trains. `meta.load_from` points a model at a saved file. Here it reads
the split from section 1 and asks for metrics and cut-offs the training run never
computed.

In [10]:
print(Path("configs/eval-checkpoints.yml").read_text())

# Evaluate the saved models on more metrics and cut-offs, without training.
# The split written in section 1 is read back, so the test set is the same.
reader:
  loading_strategy: split
  data_type: transaction
  reading_method: local
  rating_type: implicit
  split:
    local_path: ../runs/14-evaluating-existing-models/trained/ml-100k/split
    ext: .tsv
    sep: "\t"
    header: true
writer:
  dataset_name: ml-100k
  writing_method: local
  local_experiment_path: ../runs/14-evaluating-existing-models/eval-checkpoints
# BPR is built from these hyperparameters and given the saved weights, so they
# must be those it was trained with. EASE is restored whole from its file.
# num_workers: 0 builds the evaluation batches in this process, which on a
# dataset this small is much faster than starting worker processes.
models:
  BPR:
    embedding_size: 32
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 10
    learning_rate: 0.005
    meta:
      load_from: ../runs/14-evaluating-existin

With the log on, the lines to look for are `Loaded the weights of BPR` and
`Restored EASE`: the two kinds of model are loaded differently, as the next
part shows.

In [11]:
from warprec.pipelines import eval_pipeline

assert bpr.name_param in Path("configs/eval-checkpoints.yml").read_text()
logger.enable("warprec")
eval_pipeline("configs/eval-checkpoints.yml")
logger.disable("warprec")

09-10-2026 12:35:27 - WarpRec - 📝 Starting the Evaluation Pipeline.


09-10-2026 12:35:27 - WarpRec - 📝 Reading eval configuration file in: configs/eval-checkpoints.yml


09-10-2026 12:35:27 - WarpRec - 📝 Reading process completed correctly.


09-10-2026 12:35:27 - WarpRec - 📝 Setting up experiment local folder.


09-10-2026 12:35:27 - WarpRec - 📝 Experiment folder created successfully.


...

09-10-2026 12:35:27 - WarpRec - 📝 +----------+----------+----------+-----------+----------------+


09-10-2026 12:35:27 - WarpRec - 📝 | Top@50   | 0.23907  | 0.403523 | 0.0939228 |            832 |


09-10-2026 12:35:27 - WarpRec - 📝 +----------+----------+----------+-----------+----------------+


09-10-2026 12:35:27 - WarpRec - 📝 Results successfully written to ../runs/14-evaluating-existing-models/eval-checkpoints/ml-100k/evaluation/Overall_Results_09.10.2026_12-35-27.tsv


09-10-2026 12:35:27 - WarpRec - ✅ Evaluation pipeline executed successfully. WarpRec is shutting down.


The pipeline writes one `Overall_Results_<timestamp>.tsv` per run under
`<local_experiment_path>/<dataset_name>/evaluation/`, one row per model and
cut-off.

In [12]:
def results(run: str) -> pd.DataFrame:
    """The Overall_Results file of a pipeline run under RUNS."""
    (path,) = (RUNS / run / "ml-100k/evaluation").glob("Overall_Results_*.tsv")
    return pd.read_csv(path, sep="\t")


reloaded = results("eval-checkpoints")
reloaded.round(4)

,Model,Top@k,ItemCoverage,MAP,Recall,nDCG
0,BPR,10,466,0.0616,0.1064,0.1223
1,BPR,20,646,0.0616,0.1885,0.1453
2,BPR,50,927,0.0726,0.3592,0.2026
3,EASE,10,473,0.0808,0.1391,0.1542
4,EASE,20,612,0.0823,0.2357,0.1822
5,EASE,50,832,0.0939,0.4035,0.2391


At cut-off 10 these are the numbers the models had in memory. The results
file stores 16 significant digits, so the comparison is made to that
precision rather than on the rounded table:

In [13]:
at_10 = reloaded[reloaded["Top@k"] == 10].set_index("Model")[["nDCG", "Recall"]]
at_10.columns = ["nDCG@10", "Recall@10"]
difference = (at_10 - in_memory.loc[at_10.index]).abs().to_numpy().max()
print(f"largest difference from the in-memory models: {difference:.0e}")

largest difference from the in-memory models: 6e-17


### How each kind of model is loaded

An **iterative** model is built from the configuration, then given the
file's `state_dict`. The hyperparameters that shape the weights must
therefore be those it was trained with: a BPR configured with 64 dimensions
cannot take 32-dimensional embeddings.

In [14]:
try:
    build("BPR", embedding_size=64).load_state_dict(bpr_file["state_dict"])
except RuntimeError as error:
    print(str(error).splitlines()[1].strip())

size mismatch for user_embedding.weight: copying a param with shape torch.Size([943, 32]) from checkpoint, the shape in current model is torch.Size([943, 64]).


A **closed-form** model is restored whole from the file, with
`from_checkpoint` as in section 2: it is not fitted again, and the
hyperparameters in the configuration are not used. `eval-checkpoints.yml`
gives EASE `l2: 10.0`; the model evaluated above is still the saved one,
trained with `l2: 200.0`, which is why its numbers matched. A refit with
`l2: 10.0` scores differently:

In [15]:
pd.DataFrame(
    {
        "EASE restored by the pipeline (config l2 10.0)": at_10.loc["EASE"],
        "EASE fitted with l2 10.0": pd.Series(evaluate(build("EASE", l2=10.0))),
    }
).T.round(4)

,nDCG@10,Recall@10
EASE restored by the pipeline (config l2 10.0),0.1542,0.1391
EASE fitted with l2 10.0,0.1278,0.1206


### What the pipeline refuses

A checkpoint scores **indices**, not ids: row *i* of an embedding, or of
EASE's matrix, is the item with index *i* in the training run's mapping. The
pipeline therefore checks every file before evaluating it, and stops on:

- a `load_from` path that does not exist, before any data is read;
- a file holding another model than the one configured;
- a file whose user or item mappings differ from the dataset's: other data,
  another filtering or another split. (A model trained with a
  `validation_splitting` was fitted without the validation set, so the
  evaluation configuration keeps that block too.)

Each case below changes one thing in `eval-checkpoints.yml`. The third reads
the original ratings and splits them again with a 20% holdout, so the
training items differ from the checkpoint's.

In [16]:
base = yaml.safe_load(Path("configs/eval-checkpoints.yml").read_text())
ease_path = base["models"]["EASE"]["meta"]["load_from"]
other_split = {
    "reader": {
        "loading_strategy": "dataset",
        "data_type": "transaction",
        "reading_method": "local",
        "local_path": "../data/ml-100k/ratings.tsv",
        "rating_type": "implicit",
    },
    "splitter": {"test_splitting": {"strategy": "temporal_holdout", "ratio": 0.2}},
}
cases = {
    "missing file": (
        {"EASE": {"l2": 200.0, "meta": {"load_from": "no/such/file.pth"}}},
        {},
    ),
    "another model": (
        {"BPR": {**base["models"]["BPR"], "meta": {"load_from": ease_path}}},
        {},
    ),
    "other ids": ({"BPR": base["models"]["BPR"]}, other_split),
}
for name, (models, sections) in cases.items():
    config_file = RUNS / f"refused-{name.replace(' ', '-')}.yml"
    writer = {**base["writer"], "local_experiment_path": str(RUNS / "refused")}
    config_file.write_text(
        yaml.safe_dump({**base, "writer": writer, "models": models, **sections})
    )
    try:
        eval_pipeline(str(config_file))
    except (FileNotFoundError, ValueError) as error:
        print(f"{name}: {type(error).__name__}: {error}\n")

missing file: FileNotFoundError: The checkpoint of EASE (meta.load_from) does not exist: no/such/file.pth



another model: ValueError: The checkpoint ../runs/14-evaluating-existing-models/trained/ml-100k/serialized/EASE_WiseFoxOfForce.pth holds a EASE model, not a BPR.



other ids: ValueError: The checkpoint ../runs/14-evaluating-existing-models/trained/ml-100k/serialized/BPR_NippyValiantNewt.pth was trained on other item ids than this dataset: 1648 items in the checkpoint, 1613 in the dataset, 937 of the dataset's items missing from the checkpoint or at another index. Evaluate it on the data, filtering and splitting it was trained with, validation_splitting included: a model trained with a validation split was fitted without it.



The split written in section 1 is what makes the check pass: read back, it
gives the dataset the same mappings the models were trained with.

In [17]:
from warprec.utils.config.config import load_eval_configuration

eval_config = load_eval_configuration("configs/eval-checkpoints.yml")
split_dataset, _, _ = initialize_datasets(
    ReaderFactory.get_reader(eval_config), WarpRecCallback(), eval_config
)
user_map, item_map = split_dataset.get_mappings()
print(
    "same user mapping:",
    bpr_file["info"]["user_mapping"] == user_map,
    "| same item mapping:",
    bpr_file["info"]["item_mapping"] == item_map,
)

same user mapping: True | same item mapping: True


## 4. Recommendations from another framework: `ProxyRecommender`

`ProxyRecommender` is a model whose scores come from a file. The evaluator
treats it like any other model, so a list produced elsewhere gets WarpRec's
metrics, cut-offs and significance tests on WarpRec's test set.

The external model here is an item-kNN written with numpy and nothing from
WarpRec: cosine similarity between items, 50 neighbours per item, training
items removed, top 20 per user. It reads the same `train.tsv`, with the
original ids.

In [18]:
train = pd.read_csv(split_dir / "train.tsv", sep="\t")
user_ids, user_row = np.unique(train.user_id, return_inverse=True)
item_ids, item_col = np.unique(train.item_id, return_inverse=True)
X = np.zeros((len(user_ids), len(item_ids)), dtype=np.float32)
X[user_row, item_col] = 1.0

norms = np.sqrt(X.sum(axis=0))
similarity = (X.T @ X) / np.outer(norms, norms)
np.fill_diagonal(similarity, 0.0)
neighbours = np.argpartition(-similarity, 50, axis=0)[:50]
pruned = np.zeros_like(similarity)
np.put_along_axis(
    pruned, neighbours, np.take_along_axis(similarity, neighbours, axis=0), axis=0
)

scores = X @ pruned
scores[X > 0] = -np.inf
top = np.argsort(-scores, axis=1, kind="stable")[:, :20]
external = pd.DataFrame(
    {
        "user_id": np.repeat(user_ids, 20),
        "item_id": item_ids[top].ravel(),
        "rating": np.take_along_axis(scores, top, axis=1).ravel(),
    }
)
(RUNS / "external").mkdir()
external.to_csv(RUNS / "external/itemknn.tsv", sep="\t", index=False)
print(len(external), "rows")
external.head(3)

18860 rows


,user_id,item_id,rating
0,1,423,25.424221
1,1,568,24.914968
2,1,318,24.428791


The file has one row per recommended pair: user, item, score. With a header,
**the columns must be called `user_id`, `item_id` and `rating`** (whatever the
score means); without one, the first three columns are read in that order.
The ids are the original ones: `ProxyRecommender` maps them with the
dataset's mappings, which is why it takes `info`.

In [19]:
def proxy(path, header=True, ds=split_dataset):
    """A ProxyRecommender over a recommendation file, on the dataset ds."""
    return model_registry.get(
        "ProxyRecommender",
        params={"recommendation_file": str(path), "separator": "\t", "header": header},
        info=ds.info(),
    )


itemknn = proxy(RUNS / "external/itemknn.tsv")
compared = pd.DataFrame(
    {
        name: evaluate(model, split_dataset, cutoffs=(10, 20))
        for name, model in [
            ("BPR", bpr),
            ("EASE", ease),
            ("external item-kNN", itemknn),
        ]
    }
).T
compared.round(4)

,nDCG@10,Recall@10,nDCG@20,Recall@20
BPR,0.1223,0.1064,0.1453,0.1885
EASE,0.1542,0.1391,0.1822,0.2357
external item-kNN,0.1302,0.1158,0.1541,0.2012


A header named differently, `score` for instance, fails when the model is
built:

In [20]:
external.rename(columns={"rating": "score"}).to_csv(
    RUNS / "external/itemknn-score.tsv", sep="\t", index=False
)
try:
    proxy(RUNS / "external/itemknn-score.tsv")
except RuntimeError as error:
    print(str(error)[:160])

An error occurred while reading the recommendation file: Usecols do not match columns, columns expected but not found: ['rating']


### What the file leaves out, and what WarpRec does not know

`ProxyRecommender` turns the file into a sparse user × item score matrix.
Every pair the file does not list scores **0**, and the ranking is taken over
all items as for any model. Four consequences, each measured below against
the full file:

- **ids WarpRec does not know** (a user or item absent from the training
  split) are dropped silently when the file is read;
- **a user the file leaves out** is still evaluated: every item ties at 0,
  and the evaluator breaks ties per user with a seeded hash, so the user gets
  a random list (the same on every run) and a score near the random floor,
  not NaN, which pulls the mean down;
- **a list shorter than the cut-off** is padded with unlisted items, all tied
  at 0, so drawn at random for each user;
- **a negative score** ranks below every unlisted item, so the order the file
  meant is lost.

In [21]:
def write(frame: pd.DataFrame, name: str) -> Path:
    path = RUNS / "external" / f"{name}.tsv"
    frame.to_csv(path, sep="\t", index=False)
    return path


unknown = pd.DataFrame(
    {"user_id": [99999, 1], "item_id": [50, 99999], "rating": [9.0, 9.0]}
)
variants = {
    "full file": write(external, "full"),
    "+ unknown ids": write(pd.concat([external, unknown]), "unknown"),
    "users > 600 left out": write(external[external.user_id <= 600], "missing-users"),
    "top 10 only": write(external.groupby("user_id").head(10), "top10"),
    "scores - median": write(
        external.assign(rating=external.rating - external.rating.median()), "shifted"
    ),
}
pd.DataFrame(
    {
        name: {
            "rows in file": len(pd.read_csv(path, sep="\t")),
            "pairs kept": proxy(path).predictions_sparse.nnz,
            **evaluate(proxy(path), split_dataset, cutoffs=(10, 20)),
        }
        for name, path in variants.items()
    }
).T.astype({"rows in file": int, "pairs kept": int}).round(4)

,rows in file,pairs kept,nDCG@10,Recall@10,nDCG@20,Recall@20
full file,18860,18860,0.1302,0.1158,0.1541,0.2012
+ unknown ids,18862,18860,0.1302,0.1158,0.1541,0.2012
users > 600 left out,12000,12000,0.0883,0.0760,0.1015,0.1270
top 10 only,9430,9430,0.1302,0.1158,0.1172,0.1223
scores - median,18860,18860,0.0847,0.0539,0.0855,0.0796


The two unknown pairs are dropped and nothing changes. The rows for the
other cases tell where the damage is:

- leaving users out lowers every metric, because those users count with an
  random list;
- a top-10 list gives the same @10 numbers, while @20 is measured on ten
  listed items plus ten random ones;
- subtracting the median score keeps every list in the same order, but makes
  the weaker half of the scores negative, and unlisted items overtake them.

The per-user values show the left-out users directly:

In [22]:
evaluator = Evaluator(["nDCG"], [10], train_set=split_dataset.train_set.get_sparse())
evaluator.evaluate(
    proxy(variants["users > 600 left out"]),
    split_dataset.get_evaluation_dataloader(),
    "full",
    split_dataset,
)
per_user = evaluator.compute_results()[10]["nDCG"]
inverse_users, _ = split_dataset.get_inverse_mappings()
raw_user = torch.tensor([inverse_users[i] for i in range(len(per_user))])
pd.DataFrame(
    {
        "users": [(raw_user <= 600).sum().item(), (raw_user > 600).sum().item()],
        "NaN": [
            per_user[raw_user <= 600].isnan().sum().item(),
            per_user[raw_user > 600].isnan().sum().item(),
        ],
        "mean nDCG@10": [
            per_user[raw_user <= 600].nanmean().item(),
            per_user[raw_user > 600].nanmean().item(),
        ],
    },
    index=["in the file", "left out"],
).round(4)

,users,NaN,mean nDCG@10
in the file,600,0,0.1327
left out,343,0,0.0105


So an external file should list every test user, at least as many items as
the largest cut-off, and scores that are positive and larger for better items
(a reciprocal rank works when the other framework writes no score). Users who
should not be evaluated are removed from the test set, not from the file.

## 5. `meta.save_recs`: the file WarpRec writes

With `meta.save_recs: true` a pipeline writes each model's top-k list for
**every user** to `recs/<Model>_<timestamp>.tsv`, in the format
`ProxyRecommender` reads. `writer.recommendation` sets the length (`k`,
default 50), the separator, the header and the column names; leave the names
alone if the file is to be read back.

In [23]:
print(Path("configs/eval-recs.yml").read_text())

# Evaluate the saved EASE again and write its top-20 list for every user.
reader:
  loading_strategy: split
  data_type: transaction
  reading_method: local
  rating_type: implicit
  split:
    local_path: ../runs/14-evaluating-existing-models/trained/ml-100k/split
    ext: .tsv
    sep: "\t"
    header: true
writer:
  dataset_name: ml-100k
  writing_method: local
  local_experiment_path: ../runs/14-evaluating-existing-models/eval-recs
  recommendation:
    k: 20             # items per user (default 50)
    sep: "\t"
    ext: .tsv
    header: true      # user_id, item_id, rating: what ProxyRecommender reads
models:
  EASE:
    l2: 200.0
    meta:
      load_from: ../runs/14-evaluating-existing-models/trained/ml-100k/serialized/EASE_WiseFoxOfForce.pth
      save_recs: true
    optimization: {num_workers: 0}
evaluation:
  top_k: [10, 20]
  metrics: [nDCG, Recall]
  mask_seen: auto     # also decides which items the written lists may contain
general:
  device: cpu



In [24]:
eval_pipeline("configs/eval-recs.yml")
(recs_file,) = (RUNS / "eval-recs/ml-100k/recs").glob("EASE_*.tsv")
written = pd.read_csv(recs_file, sep="\t")
print(
    recs_file.name.split("_")[0],
    len(written),
    "rows,",
    written.user_id.nunique(),
    "users",
)
written.head(3)

Generating recommendation batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generating recommendation batches: 100%|██████████| 1/1 [00:00<00:00, 16.03it/s]

EASE 18860 rows, 943 users


,user_id,item_id,rating
0,1,423,0.589241
1,1,403,0.531442
2,1,568,0.517298


Read back through `ProxyRecommender`, the file gives exactly the numbers of
the EASE in memory, at both cut-offs:

In [25]:
round_trip = pd.DataFrame(
    {
        "EASE in memory": evaluate(ease, split_dataset, cutoffs=(10, 20)),
        "its file, read back": evaluate(
            proxy(recs_file), split_dataset, cutoffs=(10, 20)
        ),
    }
).T
print("identical:", round_trip.iloc[0].equals(round_trip.iloc[1]))
round_trip.round(4)

identical: True


,nDCG@10,Recall@10,nDCG@20,Recall@20
EASE in memory,0.1542,0.1391,0.1822,0.2357
"its file, read back",0.1542,0.1391,0.1822,0.2357


### `mask_seen` decides what the file contains

The lists are filtered by the same `evaluation.mask_seen` the run evaluates
with (`auto` is `pair` on data without context: every training item is
removed). With `mask_seen: none` the training items stay in, and they are
what EASE ranks highest. `write_recs` is the call behind `save_recs`; here it
writes the unmasked list directly.

In [26]:
unmasked_writer = LocalWriter(dataset_name="ml-100k", local_path=str(RUNS / "unmasked"))
unmasked_writer.write_recs(ease, split_dataset, k=20, mask_seen="none")
(unmasked_file,) = (RUNS / "unmasked/ml-100k/recs").glob("EASE_*.tsv")
unmasked = pd.read_csv(unmasked_file, sep="\t")
seen = set(zip(train.user_id, train.item_id))
share = np.mean([pair in seen for pair in zip(unmasked.user_id, unmasked.item_id)])
print(f"{share:.1%} of the rows are training items")

Generating recommendation batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generating recommendation batches: 100%|██████████| 1/1 [00:00<00:00, 17.77it/s]

82.8% of the rows are training items


Evaluated with the default `mask_seen`, those rows are removed again, so most
users are left with a few listed items and a tail of random ones:

In [27]:
pd.DataFrame(
    {
        "masked file": evaluate(proxy(recs_file), split_dataset, cutoffs=(10, 20)),
        "unmasked file": evaluate(
            proxy(unmasked_file), split_dataset, cutoffs=(10, 20)
        ),
    }
).T.round(4)

,nDCG@10,Recall@10,nDCG@20,Recall@20
masked file,0.1542,0.1391,0.1822,0.2357
unmasked file,0.0732,0.0755,0.0745,0.0819


## 6. Significance between saved and external models

The evaluation pipeline runs the same paired tests as the train pipeline
(guide 12) over every pair of models in `models`. Here: the saved BPR and EASE,
both loaded with `load_from`, and the external item-kNN.

`models` is a mapping keyed by model name, so a run holds **one**
`ProxyRecommender`, and its results are reported under that name. Two external
files, or two checkpoints of the same model, need two runs, and then no test
compares them.

In [28]:
print(Path("configs/eval-compare.yml").read_text())

# Two saved WarpRec models and one external recommendation file, evaluated
# together, with paired significance tests between every pair.
reader:
  loading_strategy: split
  data_type: transaction
  reading_method: local
  rating_type: implicit
  split:
    local_path: ../runs/14-evaluating-existing-models/trained/ml-100k/split
    ext: .tsv
    sep: "\t"
    header: true
writer:
  dataset_name: ml-100k
  writing_method: local
  local_experiment_path: ../runs/14-evaluating-existing-models/eval-compare
models:
  BPR:
    embedding_size: 32
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 10
    learning_rate: 0.005
    meta:
      load_from: ../runs/14-evaluating-existing-models/trained/ml-100k/serialized/BPR_NippyValiantNewt.pth
    optimization: {num_workers: 0}
  EASE:
    l2: 200.0
    meta:
      load_from: ../runs/14-evaluating-existing-models/trained/ml-100k/serialized/EASE_WiseFoxOfForce.pth
    optimization: {num_workers: 0}
  ProxyRecommender:   # the external file (sec

In [29]:
eval_pipeline("configs/eval-compare.yml")
results("eval-compare").round(4)

,Model,Top@k,Recall,nDCG
0,BPR,10,0.1064,0.1223
1,EASE,10,0.1391,0.1542
2,ProxyRecommender,10,0.1158,0.1302


Each test writes its own file, `<Test>_<timestamp>.tsv`, next to the results:
one row per pair of models and metric, with the p-value and the decision
before and after each correction, as `Significant` or `Not significant`.

In [30]:
(wilcoxon_file,) = (RUNS / "eval-compare/ml-100k/evaluation").glob(
    "Wilcoxon_test_*.tsv"
)
pd.read_csv(wilcoxon_file, sep="\t")

,Model A,Model B,Metric,Cutoff,Statistic,p-value,Significance (α=0.05),Significance (Holm-Bonferroni)
0,BPR,EASE,nDCG,10,54560.0,8.798200e-14,Significant,Significant
1,BPR,EASE,Recall,10,24894.5,2.108396e-11,Significant,Significant
2,EASE,ProxyRecommender,nDCG,10,63874.5,2.585946e-08,Significant,Significant
3,EASE,ProxyRecommender,Recall,10,27347.5,4.534986e-07,Significant,Significant
4,BPR,ProxyRecommender,Recall,10,37840.5,4.321747e-02,Significant,Not significant
5,BPR,ProxyRecommender,nDCG,10,75060.0,1.088266e-01,Not significant,Not significant


EASE differs significantly from both other models. The saved BPR and the
external item-kNN are closer: their Recall@10 difference passes at α = 0.05
(p = 0.043) but not after the Holm-Bonferroni correction over the six tests,
and their nDCG@10 difference is not significant.

## Summary

| Key | What it does | Docs |
|---|---|---|
| `meta.save_model` | train pipeline: write `serialized/<name_param>.pth` (weights, `info` with the id mappings, serving payload) | [Models](https://warprec.readthedocs.io/en/latest/configuration/models/) |
| `writer.save_split` | train pipeline: write `split/train.tsv` and `test.tsv`, to evaluate later on the same partition | [Writer](https://warprec.readthedocs.io/en/latest/configuration/writer/) |
| `meta.load_from` | eval pipeline: give an iterative model its saved weights, restore a closed-form one whole; a missing file, another model or other ids stop the run | [Evaluation pipeline](https://warprec.readthedocs.io/en/latest/pipelines/evaluation/) |
| `ProxyRecommender` (`recommendation_file`, `separator`, `header`) | evaluate an external `user_id, item_id, rating` file; unlisted pairs score 0, unknown ids are dropped | [Cross-framework evaluation](https://warprec.readthedocs.io/en/latest/recommenders/cross-framework-evaluation/) |
| `meta.save_recs`, `writer.recommendation` | write each user's top-k in that format, filtered by `evaluation.mask_seen` | [Writer](https://warprec.readthedocs.io/en/latest/configuration/writer/) |
| `evaluation.stat_significance` | paired tests between every pair of models in the run | [Evaluation](https://warprec.readthedocs.io/en/latest/configuration/evaluation/) |